In [5]:
# pip install optuna

In [10]:
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
import optuna
import pandas as pd
from sklearn.preprocessing import StandardScaler

In [7]:
# Load the Pima Indian Diabetes dataset (from UCI repository)
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
columns = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI',
           'DiabetesPedigreeFunction', 'Age', 'Outcome']

# Load the dataset
df = pd.read_csv(url, names=columns)

df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [8]:
import numpy as np

# Replace zero values with NaN in columns where zero is not a valid value
cols_with_missing_vals = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df[cols_with_missing_vals] = df[cols_with_missing_vals].replace(0, np.nan)

# Impute the missing values with the mean of the respective column
df.fillna(df.mean(), inplace=True)

# Check if there are any remaining missing values
print(df.isnull().sum())

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64


In [11]:
# Split into features (X) and target (y)
X = df.drop('Outcome', axis=1)
y = df['Outcome']

# Split data into training and test sets (70% train, 30% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Optional: Scale the data for better model performance
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Check the shape of the data
print(f'Training set shape: {X_train.shape}')
print(f'Test set shape: {X_test.shape}')

Training set shape: (537, 8)
Test set shape: (231, 8)


In [12]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# Define the objective function
def objective(trial):
    # Suggest values for the hyperparameters
    n_estimators = trial.suggest_int('n_estimators', 50, 200)
    max_depth = trial.suggest_int('max_depth', 3, 20)

    # Create the RandomForestClassifier with suggested hyperparameters
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42
    )

    # Perform 3-fold cross-validation and calculate accuracy
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()

    return score  # Return the accuracy score for Optuna to maximize

In [13]:
# Create a study object and optimize the objective function
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler())  # We aim to maximize accuracy
study.optimize(objective, n_trials=50)  # Run 50 trials to find the best hyperparameters

[I 2026-04-01 04:42:09,258] A new study created in memory with name: no-name-f580106d-07ab-4de4-9527-109353cd6a74
[I 2026-04-01 04:42:13,195] Trial 0 finished with value: 0.7765363128491621 and parameters: {'n_estimators': 179, 'max_depth': 14}. Best is trial 0 with value: 0.7765363128491621.
[I 2026-04-01 04:42:15,149] Trial 1 finished with value: 0.7597765363128491 and parameters: {'n_estimators': 91, 'max_depth': 10}. Best is trial 0 with value: 0.7765363128491621.
[I 2026-04-01 04:42:17,916] Trial 2 finished with value: 0.7597765363128491 and parameters: {'n_estimators': 111, 'max_depth': 3}. Best is trial 0 with value: 0.7765363128491621.
[I 2026-04-01 04:42:23,626] Trial 3 finished with value: 0.7709497206703911 and parameters: {'n_estimators': 172, 'max_depth': 18}. Best is trial 0 with value: 0.7765363128491621.
[I 2026-04-01 04:42:25,650] Trial 4 finished with value: 0.7541899441340782 and parameters: {'n_estimators': 61, 'max_depth': 10}. Best is trial 0 with value: 0.7765363

In [14]:
# Print the best result
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7802607076350093
Best hyperparameters: {'n_estimators': 127, 'max_depth': 16}


In [17]:
from sklearn.metrics import accuracy_score

# Train a RandomForestClassifier using the best hyperparameters from Optuna
best_model = RandomForestClassifier(**study.best_trial.params, random_state=42)   ## dictionary unpacking

# Fit the model to the training data
best_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Calculate the accuracy on the test set
test_accuracy = accuracy_score(y_test, y_pred)

# Print the test accuracy
print(f'Test Accuracy with best hyperparameters: {test_accuracy:.2f}')

Test Accuracy with best hyperparameters: 0.74


## Samplers in Optuna (act as Randomised Search CV)

In [18]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# Define the objective function
def objective(trial):
    # Suggest values for the hyperparameters
    n_estimators = trial.suggest_int('n_estimators', 50, 200)
    max_depth = trial.suggest_int('max_depth', 3, 20)

    # Create the RandomForestClassifier with suggested hyperparameters
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42
    )

    # Perform 3-fold cross-validation and calculate accuracy
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()

    return score  # Return the accuracy score for Optuna to maximize

In [19]:
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.RandomSampler())  # We aim to maximize accuracy
study.optimize(objective, n_trials=50)  # Run 50 trials to find the best hyperparameters

[I 2026-04-01 04:47:22,569] A new study created in memory with name: no-name-5dd6c68e-1e39-4f85-8a81-1f9cb2305333
[I 2026-04-01 04:47:23,228] Trial 0 finished with value: 0.7728119180633147 and parameters: {'n_estimators': 70, 'max_depth': 11}. Best is trial 0 with value: 0.7728119180633147.
[I 2026-04-01 04:47:27,216] Trial 1 finished with value: 0.7635009310986964 and parameters: {'n_estimators': 200, 'max_depth': 9}. Best is trial 0 with value: 0.7728119180633147.
[I 2026-04-01 04:47:30,246] Trial 2 finished with value: 0.7783985102420856 and parameters: {'n_estimators': 118, 'max_depth': 19}. Best is trial 2 with value: 0.7783985102420856.
[I 2026-04-01 04:47:31,837] Trial 3 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 58, 'max_depth': 20}. Best is trial 2 with value: 0.7783985102420856.
[I 2026-04-01 04:47:34,921] Trial 4 finished with value: 0.7709497206703911 and parameters: {'n_estimators': 93, 'max_depth': 17}. Best is trial 2 with value: 0.77839851

In [20]:
# Print the best result
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7783985102420856
Best hyperparameters: {'n_estimators': 118, 'max_depth': 19}


In [21]:
from sklearn.metrics import accuracy_score

# Train a RandomForestClassifier using the best hyperparameters from Optuna
best_model = RandomForestClassifier(**study.best_trial.params, random_state=42)

# Fit the model to the training data
best_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Calculate the accuracy on the test set
test_accuracy = accuracy_score(y_test, y_pred)

# Print the test accuracy
print(f'Test Accuracy with best hyperparameters: {test_accuracy:.2f}')

Test Accuracy with best hyperparameters: 0.74


## GridSearchCV

In [22]:
search_space = {
    'n_estimators': [50, 100, 150, 200],
    'max_depth': [5, 10, 15, 20]
}

In [23]:
# Create a study and optimize it using GridSampler
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.GridSampler(search_space))
study.optimize(objective)

[I 2026-04-01 04:49:40,638] A new study created in memory with name: no-name-48510d7c-8c61-49a1-87f2-ac9321d9e313
[I 2026-04-01 04:49:41,846] Trial 0 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 100, 'max_depth': 5}. Best is trial 0 with value: 0.7690875232774674.
[I 2026-04-01 04:49:43,348] Trial 1 finished with value: 0.7672253258845437 and parameters: {'n_estimators': 150, 'max_depth': 10}. Best is trial 0 with value: 0.7690875232774674.
[I 2026-04-01 04:49:43,913] Trial 2 finished with value: 0.7728119180633147 and parameters: {'n_estimators': 50, 'max_depth': 15}. Best is trial 2 with value: 0.7728119180633147.
[I 2026-04-01 04:49:44,933] Trial 3 finished with value: 0.7653631284916201 and parameters: {'n_estimators': 100, 'max_depth': 15}. Best is trial 2 with value: 0.7728119180633147.
[I 2026-04-01 04:49:45,984] Trial 4 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 100, 'max_depth': 20}. Best is trial 2 with value: 0.772811

In [24]:
# Print the best result
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7746741154562384
Best hyperparameters: {'n_estimators': 50, 'max_depth': 5}


In [25]:
from sklearn.metrics import accuracy_score

# Train a RandomForestClassifier using the best hyperparameters from Optuna
best_model = RandomForestClassifier(**study.best_trial.params, random_state=42)

# Fit the model to the training data
best_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Calculate the accuracy on the test set
test_accuracy = accuracy_score(y_test, y_pred)

# Print the test accuracy
print(f'Test Accuracy with best hyperparameters: {test_accuracy:.2f}')


Test Accuracy with best hyperparameters: 0.74


# Optuna Visualizations

In [26]:
# For visualizations
from optuna.visualization import plot_optimization_history, plot_parallel_coordinate, plot_slice, plot_contour, plot_param_importances

In [27]:
# 1. Optimization History
plot_optimization_history(study).show()

In [28]:
# 2. Parallel Coordinates Plot
plot_parallel_coordinate(study).show()

In [29]:
# 3. Slice Plot
plot_slice(study).show()

In [30]:
# 4. Contour Plot
plot_contour(study).show()

In [31]:
# 5. Hyperparameter Importance
plot_param_importances(study).show()

# Optimizing Multiple ML Models

In [32]:
# Importing the required libraries
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

In [33]:
# Define the objective function for Optuna
def objective(trial):
    # Choose the algorithm to tune
    classifier_name = trial.suggest_categorical('classifier', ['SVM', 'RandomForest', 'GradientBoosting'])

    if classifier_name == 'SVM':
        # SVM hyperparameters
        c = trial.suggest_float('C', 0.1, 100, log=True)
        kernel = trial.suggest_categorical('kernel', ['linear', 'rbf', 'poly', 'sigmoid'])
        gamma = trial.suggest_categorical('gamma', ['scale', 'auto'])

        model = SVC(C=c, kernel=kernel, gamma=gamma, random_state=42)

    elif classifier_name == 'RandomForest':
        # Random Forest hyperparameters
        n_estimators = trial.suggest_int('n_estimators', 50, 300)
        max_depth = trial.suggest_int('max_depth', 3, 20)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)
        bootstrap = trial.suggest_categorical('bootstrap', [True, False])

        model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            bootstrap=bootstrap,
            random_state=42
        )

    elif classifier_name == 'GradientBoosting':
        # Gradient Boosting hyperparameters
        n_estimators = trial.suggest_int('n_estimators', 50, 300)
        learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3, log=True)
        max_depth = trial.suggest_int('max_depth', 3, 20)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)

        model = GradientBoostingClassifier(
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            random_state=42
        )

    # Perform cross-validation and return the mean accuracy
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()
    return score

In [34]:
# Create a study and optimize it using CmaEsSampler
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)

[I 2026-04-01 04:54:47,199] A new study created in memory with name: no-name-039e3566-3504-48ea-8e6d-d0aaaa9727fa
[I 2026-04-01 04:54:47,326] Trial 0 finished with value: 0.6797020484171323 and parameters: {'classifier': 'SVM', 'C': 78.50439737402517, 'kernel': 'poly', 'gamma': 'scale'}. Best is trial 0 with value: 0.6797020484171323.
[I 2026-04-01 04:54:47,399] Trial 1 finished with value: 0.7709497206703911 and parameters: {'classifier': 'SVM', 'C': 0.21142917238971826, 'kernel': 'sigmoid', 'gamma': 'scale'}. Best is trial 1 with value: 0.7709497206703911.
[I 2026-04-01 04:54:53,460] Trial 2 finished with value: 0.7597765363128492 and parameters: {'classifier': 'GradientBoosting', 'n_estimators': 269, 'learning_rate': 0.02329510135261682, 'max_depth': 5, 'min_samples_split': 6, 'min_samples_leaf': 6}. Best is trial 1 with value: 0.7709497206703911.
[I 2026-04-01 04:55:00,722] Trial 3 finished with value: 0.7299813780260708 and parameters: {'classifier': 'GradientBoosting', 'n_estimat

In [35]:
# Retrieve the best trial
best_trial = study.best_trial
print("Best trial parameters:", best_trial.params)
print("Best trial accuracy:", best_trial.value)

Best trial parameters: {'classifier': 'SVM', 'C': 0.702481868377505, 'kernel': 'linear', 'gamma': 'auto'}
Best trial accuracy: 0.7858472998137803


In [36]:
study.trials_dataframe()['params_classifier'].value_counts()

,count
params_classifier,
SVM,36
GradientBoosting,8
RandomForest,6


In [37]:
study.trials_dataframe().groupby('params_classifier')['value'].mean()

,value
params_classifier,
GradientBoosting,0.743948
RandomForest,0.765363
SVM,0.772191


In [38]:
# 1. Optimization History
plot_optimization_history(study).show()

In [39]:
# 3. Slice Plot
plot_slice(study).show()

In [40]:
# 5. Hyperparameter Importance
plot_param_importances(study).show()